# Genre2Vec
Skip-gram with negative sampling, trained on genre co-occurrence using NumPy.

In [2]:
import numpy as np
import json
from collections import defaultdict

## 1. Load Data

In [3]:
def read_csv(path):
    with open(path) as f:
        lines = f.read().strip().splitlines()
    headers = [h.strip('"') for h in lines[0].split(',')]
    rows = []
    for line in lines[1:]:
        parts = [p.strip('"') for p in line.split(';')]
        if len(parts) == len(headers):
            rows.append(dict(zip(headers, parts)))
    return rows

docs = defaultdict(set)
for row in read_csv("data/csv/songs.csv"):
    docs[row["spotify_id"]].add(row["genre_name"])
documents = [list(g) for g in docs.values() if len(g) >= 2]
print(f"{len(documents):,} songs loaded")

42,280 songs loaded


## 2. Vocabulary

In [4]:
freq = defaultdict(int)
for doc in documents:
    for t in doc: freq[t] += 1

idx2token = [t for t, c in sorted(freq.items()) if c >= 3]
token2idx = {t: i for i, t in enumerate(idx2token)}
V = len(idx2token)
print(f"{V} genres in vocab")

1469 genres in vocab


## 3. Skip-gram Pairs

In [5]:
pairs = []
for doc in [[token2idx[t] for t in d if t in token2idx] for d in documents]:
    if len(doc) < 2: continue
    for i, c in enumerate(doc):
        for j in range(max(0, i-4), min(len(doc), i+5)):
            if j != i: pairs.append((c, doc[j]))

pairs = np.array(pairs, dtype=np.int32)
print(f"{len(pairs):,} training pairs")

345,794 training pairs


## 4. Train

In [6]:
D, NEG, LR, EPOCHS = 64, 5, 0.025, 5

np.random.seed(42)
Win  = (np.random.randn(V, D) * 0.01).astype(np.float32)
Wout = (np.random.randn(V, D) * 0.01).astype(np.float32)

for epoch in range(EPOCHS):
    idx = np.random.permutation(len(pairs))
    C, CTX = pairs[idx, 0], pairs[idx, 1]
    loss = 0.0

    for s in range(0, len(C), 1024):
        c, ctx = C[s:s+1024], CTX[s:s+1024]
        cur_lr = LR * max(1e-4, 1 - (epoch * len(C) + s) / (len(C) * EPOCHS))

        vc, vo = Win[c], Wout[ctx]
        p = 1 / (1 + np.exp(-np.clip((vc * vo).sum(1), -20, 20)))
        g = (p - 1)[:, None]
        loss += -np.log(p + 1e-9).sum()
        np.add.at(Win,  c,   -cur_lr * g * vo)
        np.add.at(Wout, ctx, -cur_lr * g * vc)

        ns = np.random.randint(0, V, (len(c), NEG))
        for k in range(NEG):
            n = ns[:, k]; vn = Wout[n]
            pn = 1 / (1 + np.exp(-np.clip((Win[c] * vn).sum(1), -20, 20)))
            gn = pn[:, None]
            loss += -np.log(1 - pn + 1e-9).sum()
            np.add.at(Win,  c, -cur_lr * gn * vn)
            np.add.at(Wout, n, -cur_lr * gn * Win[c])

    print(f"Epoch {epoch+1}/{EPOCHS} — loss={loss/len(C):.4f}")

np.save("genre2vec_embeddings.npy", Win)
with open("data/embeddings/genre2vec_vocab.json", "w") as f:
    json.dump({"token2idx": token2idx, "idx2token": idx2token}, f)
print("Saved.")

Epoch 1/5 — loss=2.6603
Epoch 2/5 — loss=0.9665
Epoch 3/5 — loss=0.7006
Epoch 4/5 — loss=0.6197
Epoch 5/5 — loss=0.5883
Saved.


## 5. Nearest Neighbours

In [7]:
def most_similar(token, topk=10):
    idx = token2idx[token]
    q = Win[idx]
    norms = np.linalg.norm(Win, axis=1)
    sims = Win @ q / (norms * np.linalg.norm(q) + 1e-9)
    sims[idx] = -1
    top = np.argsort(sims)[::-1][:topk]
    return [(idx2token[i], float(sims[i])) for i in top]

for name, sim in most_similar("pop rap"):
    print(f"{name:<30} {sim:.3f}")

dwn trap                       0.990
rap                            0.983
southern hip hop               0.983
trap music                     0.978
bounce                         0.971
drill                          0.965
dirty south rap                0.960
crunk                          0.954
slow game                      0.954
deep trap                      0.947
